# ForenSight — Test AIDE (original full) trên B-Free

## Mục đích

Chạy **cùng 11 tập B-Free** như `clip_bfree`, nhưng đổi model:
**CLIP ViT-L/14 + probe** → **AIDE original full** (893M tham số).

Để so sánh trực tiếp: cùng dữ liệu, cùng metric, khác model.

## Model

`model_trainable.pt` — checkpoint canonical của AIDE gốc Full.

| | |
|---|---|
| kiến trúc | OpenCLIP **ConvNeXt-XXLarge** (frozen) + DCT/SRM dual-ResNet50 |
| tham số | 893,734,732 (50,443,062 trainable) |
| val balanced accuracy | **0.9264** (epoch 1) |
| SHA-256 | `62848895...44e47c3` — đã kiểm khớp |
| train trên | `tiny_genimage` (7 generator) |

## ⚠️ Điểm khác biệt so với notebook CommFor gốc

Notebook gốc `test_..._commfor_exact_kaggle.ipynb` test trên **CommFor** (1000 ảnh,
cohort cố định, SHA-256 manifest khóa cứng). Notebook này test trên **B-Free**.

Giữ nguyên: cách build model, cách load checkpoint, cách chuẩn bị input.
Thay: nguồn ảnh (CommFor → B-Free) và bộ metric (dùng của `clip_bfree`).

## Input AIDE cần gì

AIDE **không ăn 1 ảnh** — nó ăn **5 kênh** `[5,3,256,256]`:

```
ảnh → DCT cắt ô 32×32 stride 16 → chấm điểm 6 dải chéo
    → chọn 4 ô (min1/max1/min2/max2) + ảnh gốc = 5 kênh
```

## ⚠️ Cảnh báo về T7–T9

Giống `clip_bfree`: RAISE real **không vuông** (1258×833), fake **1024×1024 vuông**.
AIDE cắt ô theo **pixel gốc** → số ô và nội dung ô khác hẳn nhau.

=> AUC T7–T9 **có thể chỉ phản ánh tỉ lệ khung hình**, không phải dấu vết AI.

In [1]:
# ============================================================== CẤU HÌNH

PER_CAT = 30            # so anh moi category
SEED = 42
BATCH_SIZE = 1          # AIDE goc dung batch 1
NUM_WORKERS = 0

BFREE_NAME = "b-free-training-data"
AIDE_CKPT_NAME = "model_trainable.pt"

RUN_NAME = f"aide_bfree_seed{SEED}"
print(f"RUN_NAME : {RUN_NAME}")
print(f"PER_CAT  : {PER_CAT}")

RUN_NAME : aide_bfree_seed42
PER_CAT  : 30


In [2]:
# ============================================================== CAI DEPENDENCIES
# Copy NGUYEN tu notebook goc cua AIDE (cell 1 cua test_..._commfor_exact).
# THIEU cell nay => ModuleNotFoundError: No module named 'open_clip'
# (bai hoc E33: bo sot cell cai dat khi tai dung notebook khac).
%pip install -q "open_clip_torch==2.26.1" "pandas<3.0" "scikit-learn<1.9" "pillow<12.0" "tqdm"

# ---------------------------------------------------------- HF_TOKEN
# Copy tu notebook goc AIDE: tai ConvNeXt 3.4GB tu HuggingFace de bi rate-limit
# neu khong co token. Tao Kaggle Secret ten "HF_TOKEN" de dung.
import os
try:
    from kaggle_secrets import UserSecretsClient
    _t = UserSecretsClient().get_secret("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
        os.environ["HUGGING_FACE_HUB_TOKEN"] = _t
        print("HF_TOKEN: da nap tu Kaggle Secrets.")
except Exception:
    print("HF_TOKEN: khong cau hinh — tai cong khai (co the cham).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
HF_TOKEN: khong cau hinh — tai cong khai (co the cham).


## 1. Import & đường dẫn

In [3]:
import contextlib, gc, json, math, os, shutil, sys, time
from collections import defaultdict
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from torch.utils.data import DataLoader, Dataset

plt.rcParams.update({"figure.autolayout": True, "font.size": 10})

# ============================================================ DUONG DAN
# HARD-CODE TUAN (bai hoc E28): chi 1 lenh stat, khong quet de quy.
P_BFREE = Path("/kaggle/input/datasets/baopv051/b-free-training-data")
P_AIDE_DS = Path("/kaggle/input/aide-original-full")


def pick_first(cands, check=None):
    for c in cands:
        c = Path(c)
        if check is None:
            if c.exists():
                return c
        elif check(c):
            return c
    return None


def find_slug(slug, container):
    root = Path(container)
    if not root.is_dir():
        return None
    try:
        for owner in root.iterdir():
            if not owner.is_dir():
                continue
            cand = owner / slug
            if cand.is_dir():
                return cand
    except (PermissionError, OSError):
        pass
    return None


t0 = time.time()
BFREE = pick_first([P_BFREE, Path("/kaggle/input") / BFREE_NAME],
                   check=lambda p: p.is_dir())
if BFREE is None:
    BFREE = find_slug(BFREE_NAME, "/kaggle/input/datasets")
print(f"  [BFREE]   {time.time()-t0:.2f}s")

t0 = time.time()
CKPT = pick_first([P_AIDE_DS / AIDE_CKPT_NAME,
                   Path("/kaggle/input") / AIDE_CKPT_NAME,
                   Path.cwd() / AIDE_CKPT_NAME], check=lambda p: p.is_file())
if CKPT is None:
    for hit in Path("/kaggle/input").glob(f"**/{AIDE_CKPT_NAME}"):
        CKPT = hit
        break
print(f"  [CKPT]    {time.time()-t0:.2f}s -> {CKPT}")

# File .py goc cua AIDE (de import kien truc — KHONG copy-paste)
AIDE_TRAIN_PY = "train_aide_original_full_tiny_commfor_kaggle.py"
t0 = time.time()
AIDE_SRC = pick_first([P_AIDE_DS / AIDE_TRAIN_PY,
                       Path("/kaggle/input") / AIDE_TRAIN_PY,
                       Path.cwd() / AIDE_TRAIN_PY], check=lambda p: p.is_file())
if AIDE_SRC is None:
    for hit in Path("/kaggle/input").glob(f"**/{AIDE_TRAIN_PY}"):
        AIDE_SRC = hit
        break
print(f"  [AIDE_SRC]{time.time()-t0:.2f}s -> {AIDE_SRC}")

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
OUT_DIR = (Path("/kaggle/working/results") / RUN_NAME
           if Path("/kaggle/working").is_dir()
           else Path("results") / RUN_NAME)
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"BFREE    : {BFREE}")
print(f"CKPT     : {CKPT}")
print(f"AIDE_SRC : {AIDE_SRC}")
print(f"DEVICE   : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"OUT_DIR  : {OUT_DIR}")

assert BFREE is not None, f"Khong tim thay {BFREE_NAME}"
assert CKPT is not None, f"Khong tim thay {AIDE_CKPT_NAME}"
assert AIDE_SRC is not None, f"Khong tim thay {AIDE_TRAIN_PY}"

  [BFREE]   0.00s
  [CKPT]    0.01s -> /kaggle/input/datasets/truongnhatnguyen2805/aide-original-full/model_trainable.pt
  [AIDE_SRC]0.00s -> /kaggle/input/datasets/truongnhatnguyen2805/aide-original-full/train_aide_original_full_tiny_commfor_kaggle.py
BFREE    : /kaggle/input/datasets/baopv051/b-free-training-data
CKPT     : /kaggle/input/datasets/truongnhatnguyen2805/aide-original-full/model_trainable.pt
AIDE_SRC : /kaggle/input/datasets/truongnhatnguyen2805/aide-original-full/train_aide_original_full_tiny_commfor_kaggle.py
DEVICE   : cuda
GPU      : Tesla T4
OUT_DIR  : /kaggle/working/results/aide_bfree_seed42


## 2. Nạp AIDE (import nguyên file gốc)

In [4]:
# ============================================================== NAP AIDE
# KHONG copy-paste kien truc — import NGUYEN file .py goc cua AIDE.
# Ly do: copy-paste se LECH voi checkpoint khi sua nham (bai hoc E25-E32).
# File goc co `if __name__ == "__main__"` => import sach.

import importlib.util

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
shutil.copy(str(AIDE_SRC), str(WORK / AIDE_TRAIN_PY))
sys.path.insert(0, str(WORK))

_spec = importlib.util.spec_from_file_location("aide_src", str(WORK / AIDE_TRAIN_PY))
aide = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(aide)
sys.modules["aide_src"] = aide

print("  import AIDE OK")
print(f"  IMAGENET_MEAN = {aide.IMAGENET_MEAN.flatten().tolist()}")
print(f"  IMAGENET_STD  = {aide.IMAGENET_STD.flatten().tolist()}")
print(f"  co build_model: {hasattr(aide, 'build_model')}")
print(f"  co load_trainable_model_state: {hasattr(aide, 'load_trainable_model_state')}")

  import AIDE OK
  IMAGENET_MEAN = [0.48500001430511475, 0.4560000002384186, 0.4059999883174896]
  IMAGENET_STD  = [0.2290000021457672, 0.2240000069141388, 0.22499999403953552]
  co build_model: True
  co load_trainable_model_state: True


## 3. Lấy mẫu B-Free — 30 ảnh/category

In [5]:
# ============================================================== LAY MAU B-FREE
import csv, os

CSV_TRAIN = BFREE / "train_list.csv"
rows = list(csv.DictReader(open(CSV_TRAIN, encoding="utf-8", errors="replace")))
print(f"  train_list.csv: {len(rows)} dong")

# Nhom id theo same_category
by_cat = defaultdict(list)
for r in rows:
    by_cat[r["same_category"]].append(r["id"])

rng = np.random.default_rng(SEED)
picked_ids = []
for cat in sorted(by_cat):
    ids = by_cat[cat]
    n = min(PER_CAT, len(ids))
    idx = rng.permutation(len(ids))[:n]
    for i in idx:
        picked_ids.append(ids[i])

print(f"  category          : {len(by_cat)}")
print(f"  id duoc chon      : {len(picked_ids)}")
print(f"  category du 30    : {sum(1 for c in by_cat if len(by_cat[c])>=PER_CAT)}/{len(by_cat)}")

# Cac thu muc
def pick(root, *cands):
    """Tra ve duong dan DAU TIEN ton tai trong cac ung vien.

    Bai hoc (ISS-10): ban truoc viet `resolve_dir(BFREE,"latent-diffusion")
    / "latent-diffusion"`. Nhung resolve_dir DA tra ve duong dan long doi
    => noi them => thanh 3 tang => SAI, tap ldm se rong.
    Cach dung: liet ke TAT CA ung vien, lay cai dau tien ton tai.
    """
    for c in cands:
        p = root / c
        if p.is_dir():
            return p
    return root / cands[0]      # khong co -> tra ung vien dau de bao loi ro


D = {
    "real_coco":   pick(BFREE, "COCO_real_512/COCO_real_512", "COCO_real_512"),
    "selfcond":    pick(BFREE, "SD2.1_selfconditioned/SD2.1_selfconditioned",
                        "SD2.1_selfconditioned"),
    "selfcond_bg": pick(BFREE, "SD2.1_selfconditioned_origBG/SD2.1_selfconditioned_origBG",
                        "SD2.1_selfconditioned_origBG"),
    "samecat":     pick(BFREE, "SD2.1_inpainted_samecat/SD2.1_inpainted_samecat",
                        "SD2.1_inpainted_samecat"),
    "samecat_bg":  pick(BFREE, "SD2.1_inpainted_samecat_origBG/SD2.1_inpainted_samecat_origBG",
                        "SD2.1_inpainted_samecat_origBG"),
    "diffcat":     pick(BFREE, "SD2.1_inpainted_diffcat/SD2.1_inpainted_diffcat",
                        "SD2.1_inpainted_diffcat"),
    "diffcat_bg":  pick(BFREE, "SD2.1_inpainted_diffcat_origBG/SD2.1_inpainted_diffcat_origBG",
                        "SD2.1_inpainted_diffcat_origBG"),
    "flux":        pick(BFREE, "sd3_flux/flux"),
    "sd3":         pick(BFREE, "sd3_flux/sd3_large"),
    "ldm":         pick(BFREE, "latent-diffusion/latent-diffusion", "latent-diffusion"),
    "raise":       pick(BFREE, "real_RAISE_1k/real_RAISE_1k", "real_RAISE_1k"),
}

# KIEM CHUNG: moi duong dan PHAI ton tai (khong doan)
_missing = [k for k, v in D.items() if not v.is_dir()]
if _missing:
    print(f"  [CANH BAO] {len(_missing)} thu muc KHONG ton tai: {_missing}")
    print(f"     BFREE = {BFREE}")
    for k in _missing:
        print(f"       {k:12s} -> {D[k]}")
else:
    print(f"  [OK] ca {len(D)} thu muc deu ton tai")

# E28b: KHONG dem so file o day. `len([p for p in d.iterdir() if p.is_file()])`
# tren 51,517 file x 7 thu muc = ~360k syscall stat qua FUSE => mat ~950s (16 phut)
# chi de IN RA MAN HINH. Vong lap that o duoi (list_files) da dem roi.
print()
print(f"  {'nhom':12s} {'ton tai':>8s}")
for k, d in D.items():
    print(f"  {k:12s} {str(d.is_dir()):>8s}")


def find_by_id(d, sid):
    """Tim file theo id 12 chu so. Boc try: id co the khong phai so (ISS-8)."""
    try:
        p = d / f"{int(sid):012d}.png"
        if p.is_file():
            return p
    except (ValueError, TypeError):
        pass
    p2 = d / f"{sid}.png"
    return p2 if p2.is_file() else None


IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}


def list_files(d):
    """CHI lay file ANH — bo qua .pdf, .csv, .txt, file an, v.v.

    Bai hoc: lan chay dau that bai vi `RAISE_License.pdf` bi coi la anh.
    Dung os.scandir (nhanh hon iterdir+is_file vi tan dung cache cua OS).
    """
    if not d.is_dir():
        return []
    out = []
    with os.scandir(d) as it:
        for e in it:
            if not e.is_file():
                continue
            nm = e.name
            if nm.startswith("."):
                continue
            if Path(nm).suffix.lower() in IMG_EXT:
                out.append(Path(e.path))
    return sorted(out)


# ---- Tap COCO + SD2.1: ghep theo ID ----
# Ghep cap THEO ID (khong theo index) — E30: ghep theo index se LECH CAP
# neu mot thu muc fake thieu vai file (real[i] khong con tuong ung fake[i]).
coco_real, sd21 = [], {}
PAIRS_BY_ID = {}          # key -> [(real_path, fake_path), ...] da ghep dung ID
for sid in picked_ids:
    pr = find_by_id(D["real_coco"], sid)
    if pr is None:
        continue
    coco_real.append(pr)
    for key in ("selfcond", "selfcond_bg", "samecat", "samecat_bg",
                "diffcat", "diffcat_bg"):
        pf = find_by_id(D[key], sid)
        if pf is not None:
            sd21.setdefault(key, []).append(pf)
            PAIRS_BY_ID.setdefault(key, []).append((pr, pf))

# KIEM CHUNG ghep cap: moi cap phai CUNG STEM (id)
_bad_pair = {}
for key, pairs in PAIRS_BY_ID.items():
    bad = sum(1 for r, f in pairs if Path(r).stem != Path(f).stem)
    if bad:
        _bad_pair[key] = bad
if _bad_pair:
    print(f"  [CANH BAO] ghep cap LECH ID: {_bad_pair}")
else:
    print(f"  [OK] ghep cap theo ID dung cho ca "
          f"{len(PAIRS_BY_ID)} nhom fake")

print()
print(f"  COCO real         : {len(coco_real)}")
for k in ("selfcond", "selfcond_bg", "samecat", "samecat_bg", "diffcat", "diffcat_bg"):
    print(f"  {k:17s} : {len(sd21.get(k, []))}")

# GUARD: neu ghep ID that bai (vd BFREE sai) thi coco_real rong => 9 tap T1-T6
# se bi cell 9 `continue` BO QUA AM THAM, va summary.json se gan rong ma khong
# bao loi ro. Phai DUNG NGAY tai day.
if len(coco_real) < 20:
    raise SystemExit(
        f"GHEP CAP THAT BAI: chi {len(coco_real)} anh COCO.\n"
        f"  BFREE   = {BFREE}\n"
        f"  real_coco = {D['real_coco']}  (ton tai: {D['real_coco'].is_dir()})\n"
        f"  => Kiem duong dan. KHONG chay tiep vi ket qua se rong."
    )
_n_sd21 = min((len(sd21.get(k, [])) for k in
               ("selfcond", "selfcond_bg", "samecat", "samecat_bg", "diffcat", "diffcat_bg")),
              default=0)
if _n_sd21 < 20:
    raise SystemExit(f"GHEP CAP THAT BAI: chi {_n_sd21} anh SD2.1. Kiem duong dan.")

# ---- Tap RAISE + gen moi: ghep theo TEN FILE ----
# Bai hoc (E23): ban cu ghep theo THU TU va ghi "ten file khac nhau" — SAI.
# Log thuc te cho thay flux[0] va RAISE[0] DEU la "r000da54ft.png" => cung ten.
# Ghep theo thu tu la dung NHUNG phai kiem chung, khong duoc doan.
raise_files = list_files(D["raise"])
flux_files = list_files(D["flux"])
sd3_files = list_files(D["sd3"])
ldm_files = list_files(D["ldm"])
# E32: T7-T9 KHONG co guard truoc do => neu flux/sd3/ldm sai duong dan thi
# cell 9 `continue` BO QUA AM THAM, bang ket qua thieu 3 tap ma khong bao loi.
if min(len(raise_files), len(flux_files), len(sd3_files), len(ldm_files)) < 20:
    raise SystemExit(
        f"T7-T9 THAT BAI: raise={len(raise_files)} flux={len(flux_files)} "
        f"sd3={len(sd3_files)} ldm={len(ldm_files)}. Kiem duong dan sd3_flux / "
        f"latent-diffusion / real_RAISE_1k."
    )
print()
print(f"  RAISE             : {len(raise_files)}  (chi file ANH)")
print(f"  flux / sd3 / ldm  : {len(flux_files)} / {len(sd3_files)} / {len(ldm_files)}")
if flux_files:
    print(f"    vd ten flux     : {flux_files[0].name}")
if raise_files:
    print(f"    vd ten RAISE    : {raise_files[0].name}")

# Ghep theo thu tu (so luong gan bang nhau)
# Kiem chung: ten file co khop khong? — PHAI kiem CA 3 (ban cu chi kiem flux).
_n = min(len(raise_files), len(flux_files), len(sd3_files), len(ldm_files))
pairing = {}
for key, lst in (("flux", flux_files), ("sd3", sd3_files), ("ldm", ldm_files)):
    k = sum(1 for i in range(_n) if raise_files[i].name == lst[i].name)
    pairing[key] = {"khop": k, "tong": _n, "ti_le": k / _n if _n else 0.0}
    print(f"  khop ten RAISE<->{key:4s}: {k}/{_n}  ({k/_n*100:.1f}%)" if _n else "")
ALL_OK = all(v["khop"] == _n for v in pairing.values())
if ALL_OK:
    print("  => CA 3 CUNG TEN FILE. Ghep theo thu tu la AN TOAN (da kiem chung ten).")
else:
    print("  => CO tap KHAC TEN. Ghep theo thu tu CHI la gia dinh — da ghi vao limitations.")
print("  LUU Y: khop TEN khong chung minh fake sinh TU chinh anh real do.")
print("         Voi T1-T6 tinh chat nay duoc dam bao boi thiet ke B-Free; T7-T9 thi khong.")

# Ghep theo thu tu
n_new = _n
rng2 = np.random.default_rng(SEED + 1)
sel = rng2.permutation(n_new)[:min(n_new, 1000)]
raise_sel = [raise_files[i] for i in sel]
new_gen = {
    "flux": [flux_files[i] for i in sel],
    "sd3":  [sd3_files[i] for i in sel],
    "ldm":  [ldm_files[i] for i in sel],
}
print(f"  ghep theo thu tu  : {len(raise_sel)} cap")

  train_list.csv: 51517 dong
  category          : 80
  id duoc chon      : 2217
  category du 30    : 69/80
  [OK] ca 11 thu muc deu ton tai

  nhom          ton tai
  real_coco        True
  selfcond         True
  selfcond_bg      True
  samecat          True
  samecat_bg       True
  diffcat          True
  diffcat_bg       True
  flux             True
  sd3              True
  ldm              True
  raise            True
  [OK] ghep cap theo ID dung cho ca 6 nhom fake

  COCO real         : 2217
  selfcond          : 2217
  selfcond_bg       : 2217
  samecat           : 2217
  samecat_bg        : 2217
  diffcat           : 2217
  diffcat_bg        : 2217

  RAISE             : 1000  (chi file ANH)
  flux / sd3 / ldm  : 1000 / 1000 / 1000
    vd ten flux     : r000da54ft.png
    vd ten RAISE    : r000da54ft.png
  khop ten RAISE<->flux: 1000/1000  (100.0%)
  khop ten RAISE<->sd3 : 1000/1000  (100.0%)
  khop ten RAISE<->ldm : 1000/1000  (100.0%)
  => CA 3 CUNG TEN FILE. Ghep theo th

## 4. Trích điểm AIDE

In [6]:
# ============================================================== NAP WEIGHTS + TRICH DIEM
# ConvNeXt-XXLarge (frozen) KHONG nam trong model_trainable.pt -> tai rieng.
# (Can bat internet: enable_internet=true)

t0 = time.time()
ckpt = torch.load(str(CKPT), map_location="cpu", weights_only=False)
if not isinstance(ckpt, dict) or "model_trainable_state_dict" not in ckpt:
    raise TypeError("Checkpoint khong dung dinh dang AIDE trainable-state")
if ckpt.get("architecture") != (
        "AIDE full hybrid: OpenCLIP ConvNeXt-XXLarge + DCT/SRM dual ResNet50"):
    raise ValueError(f"Kien truc khac: {ckpt.get('architecture')}")

cfg = dict(ckpt["config"])
IMAGE_SIZE = int(cfg["image_size"])
print(f"  checkpoint: epoch={ckpt.get('epoch')}  "
      f"val_ba={ckpt.get('best_val_balanced_accuracy'):.4f}")
print(f"  image_size={IMAGE_SIZE}  dct={cfg['dct_window_size']}/"
      f"{cfg['dct_stride']}/{cfg['dct_grade_bands']}")
print(f"  tham so trainable: "
      f"{sum(t.numel() for t in ckpt['model_trainable_state_dict'].values()):,}")

# ---------------------------------------------------------- model_config
model_config = dict(aide.DEFAULT_CONFIG)
model_config.update({
    "input_root": "/kaggle/input",
    "random_seed": SEED,
    "dct_window_size": int(cfg["dct_window_size"]),
    "dct_stride": int(cfg["dct_stride"]),
    "dct_grade_bands": int(cfg["dct_grade_bands"]),
    "image_size": IMAGE_SIZE,
    "semantic_model_name": str(cfg["semantic_model_name"]),
    "semantic_pretrained_tag": str(cfg["semantic_pretrained_tag"]),
    "semantic_checkpoint": None,          # None => tai tu HF theo tag
    "semantic_half_precision": bool(cfg["semantic_half_precision"]),
    "imagenet_resnet_init": False,        # checkpoint da co san ResNet
    "resnet_checkpoint": None,
})

print("\nDang build AIDE (tai ConvNeXt-XXLarge ~3.4GB neu chua co)...")
model, sem_prov = aide.build_model(model_config, DEVICE)
print(f"  build xong ({time.time()-t0:.1f}s)")
print(f"  semantic: {sem_prov.get('pretrained')}")

t0 = time.time()
aide.load_trainable_model_state(model, ckpt["model_trainable_state_dict"])
model.eval()
del ckpt
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"  load state_dict OK ({time.time()-t0:.1f}s)")

# ---------------------------------------------------------- dataset
SELECTOR = aide.AIDEDCTPatchSelector(
    window_size=int(cfg["dct_window_size"]),
    stride=int(cfg["dct_stride"]),
    grade_bands=int(cfg["dct_grade_bands"]))
I_MEAN, I_STD = aide.IMAGENET_MEAN, aide.IMAGENET_STD


class AIDEDS(Dataset):
    """Tra ve (inputs[5,3,S,S], label, bad, size[w,h]) — giong ImageDS clip_bfree."""

    def __init__(self, paths, labels):
        self.paths, self.labels = paths, labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            with Image.open(self.paths[i]) as im:
                im = im.convert("RGB")
                w, h = im.size
                arr = np.asarray(im, dtype=np.float32).copy() / 255.0
                tensor = torch.from_numpy(arr).permute(2, 0, 1).contiguous()
            patches, _sel = SELECTOR(tensor)
            patches = F.interpolate(patches, size=(IMAGE_SIZE, IMAGE_SIZE),
                                    mode="bilinear", align_corners=False, antialias=True)
            raw = F.interpolate(tensor.unsqueeze(0), size=(IMAGE_SIZE, IMAGE_SIZE),
                                mode="bilinear", align_corners=False, antialias=True)
            patches = (patches - I_MEAN) / I_STD
            raw = (raw - I_MEAN) / I_STD
            model_input = torch.cat([patches, raw], dim=0)      # [5,3,S,S]
            return (model_input, torch.tensor([float(self.labels[i])]),
                    torch.tensor([0.0]), torch.tensor([w, h], dtype=torch.long))
        except Exception:
            blank = torch.zeros(5, 3, IMAGE_SIZE, IMAGE_SIZE)
            return (blank, torch.tensor([float(self.labels[i])]),
                    torch.tensor([1.0]), torch.tensor([0, 0], dtype=torch.long))


def feats_of(paths, labels, tag=""):
    """Tra ve DIEM FAKE (N,) + kich thuoc + path, DA LOC file hong."""
    ld = DataLoader(AIDEDS(paths, labels), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=NUM_WORKERS)
    scores, bad_flags, sizes, keep = [], [], [], []
    t0 = time.time()
    seen = 0
    with torch.inference_mode():
        for px, _y, bad, sz in ld:
            amp = (torch.autocast(device_type="cuda", dtype=torch.float16)
                   if DEVICE.type == "cuda" else contextlib.nullcontext())
            with amp:
                logits = model(px.to(DEVICE))
            prob = torch.softmax(logits.float(), dim=-1)[:, 1]
            scores.append(prob.cpu().numpy())
            bad_flags.append(bad.view(-1).numpy())
            sizes.extend([tuple(int(v) for v in row) for row in sz.tolist()])
            k = len(px)
            keep.extend(paths[seen:seen + k])
            seen += k
    S = np.concatenate(scores)
    bad = np.concatenate(bad_flags).astype(bool)
    n_bad = int(bad.sum())
    if n_bad:
        print(f"    [CANH BAO] {tag}: {n_bad} file hong, da loai")
    S = S[~bad]
    sizes = [s for s, b in zip(sizes, bad) if not b]
    keep = [p for p, b in zip(keep, bad) if not b]
    labels = [l for l, b in zip(labels, bad) if not b]
    print(f"    {tag:34s} S={S.shape}  ({time.time()-t0:.1f}s)")
    return S, sizes, keep, labels


# ---- 9 tap real-vs-fake + 2 tap chi-real ----
SETS = []
for _name, _key in (("T1_selfcond", "selfcond"),
                    ("T2_selfcond_bg", "selfcond_bg"),
                    ("T3_samecat", "samecat"),
                    ("T4_samecat_bg", "samecat_bg"),
                    ("T5_diffcat", "diffcat"),
                    ("T6_diffcat_bg", "diffcat_bg")):
    _pairs = PAIRS_BY_ID.get(_key, [])
    if _pairs:
        SETS.append((_name, [p[0] for p in _pairs], [p[1] for p in _pairs]))
SETS += [
    ("T7_flux", raise_sel, new_gen["flux"]),
    ("T8_sd3", raise_sel, new_gen["sd3"]),
    ("T9_ldm", raise_sel, new_gen["ldm"]),
]

print()
print("Trich diem AIDE (AIDE CHAM hon CLIP nhieu — kien nhan)...")
DATA = {}
for name, reals, fakes in SETS:
    if not reals or not fakes:
        print(f"  BO QUA {name}: real={len(reals)} fake={len(fakes)}")
        continue
    n = min(len(reals), len(fakes))
    paths = [str(p) for p in reals[:n]] + [str(p) for p in fakes[:n]]
    labels = [0] * n + [1] * n
    S, sizes, keep, labels = feats_of(paths, labels, name)
    n_pc = len(labels) // 2
    DATA[name] = {"F": S.reshape(-1, 1), "y": np.array(labels), "n_per_class": n_pc,
                  "n": len(labels), "sizes": sizes, "paths": keep}

for name, src, tag in (("R_coco", coco_real, "COCO real-only"),
                       ("R_raise", raise_sel, "RAISE real-only")):
    sp = [str(p) for p in src]
    if not sp:
        print(f"  BO QUA {tag}: khong co file")
        continue
    S, sizes, keep, _ = feats_of(sp, [0] * len(sp), tag)
    DATA[name] = {"F": S.reshape(-1, 1), "y": np.zeros(len(keep), int), "n": len(keep),
                  "n_per_class": len(keep), "sizes": sizes, "paths": keep}

print(f"\n  xong. {len(DATA)} tap.")

# ---- LUU CACHE ----
np.savez_compressed(
    OUT_DIR / "scores.npz",
    **{f"{k}_S": v["F"].ravel() for k, v in DATA.items()},
    **{f"{k}_y": v["y"] for k, v in DATA.items()},
)
json.dump({k: {"n": v["n"], "n_per_class": v["n_per_class"],
               "sizes": [list(s) for s in v["sizes"]], "paths": v["paths"]}
           for k, v in DATA.items()},
          open(OUT_DIR / "bfree_index.json", "w"), indent=2)
print(f"  da luu cache: {OUT_DIR}/scores.npz")

  checkpoint: epoch=1  val_ba=0.9264
  image_size=256  dct=32/16/6
  tham so trainable: 50,443,062

Dang build AIDE (tai ConvNeXt-XXLarge ~3.4GB neu chua co)...


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading OpenCLIP convnext_xxlarge pretrained=laion2b_s34b_b82k_augreg_soup


open_clip_pytorch_model.bin:   0%|          | 0.00/4.80G [00:00<?, ?B/s]

  build xong (1146.1s)
  semantic: laion2b_s34b_b82k_augreg_soup
  load state_dict OK (0.4s)

Trich diem AIDE (AIDE CHAM hon CLIP nhieu — kien nhan)...
    T1_selfcond                        S=(4434,)  (669.0s)
    T2_selfcond_bg                     S=(4434,)  (502.9s)
    T3_samecat                         S=(4434,)  (496.4s)
    T4_samecat_bg                      S=(4434,)  (492.6s)
    T5_diffcat                         S=(4434,)  (535.5s)
    T6_diffcat_bg                      S=(4434,)  (492.5s)
    T7_flux                            S=(2000,)  (578.0s)
    T8_sd3                             S=(2000,)  (488.6s)
    T9_ldm                             S=(2000,)  (305.7s)
    COCO real-only                     S=(2217,)  (214.6s)
    RAISE real-only                    S=(1000,)  (228.0s)

  xong. 11 tap.
  da luu cache: /kaggle/working/results/aide_bfree_seed42/scores.npz


## 5. Đánh giá đầy đủ metric

In [7]:
# ============================================================== DANH GIA
def full_metrics(y, s, thr):
    pred = (s > thr).astype(int)
    real, fake = y == 0, y == 1
    cm = confusion_matrix(y, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "threshold": float(thr),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(0.5 * (accuracy_score(y[real], pred[real])
                                          + accuracy_score(y[fake], pred[fake]))),
        "real_recall": float(recall_score(y, pred, pos_label=0, zero_division=0)),
        "fake_recall": float(recall_score(y, pred, pos_label=1, zero_division=0)),
        "fake_precision": float(precision_score(y, pred, pos_label=1, zero_division=0)),
        "fake_f1": float(f1_score(y, pred, pos_label=1, zero_division=0)),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "confusion_matrix": cm.tolist(),
    }


def best_thr(y, s):
    fpr, tpr, thrs = roc_curve(y, s)
    ba = (tpr + (1 - fpr)) / 2
    i = int(np.argmax(ba))
    return float(thrs[i]) if i < len(thrs) else 0.5


THR_FIXED = 0.5     # nguong MAC DINH 0.5

# Nguong hieu chinh DUNG CACH: chon tren VAL cua tiny_genimage (ImageNet),
# KHONG chon tren tap test B-Free. Day moi la "khong tu cho diem minh".
# AIDE: KHONG co F_va/y_va (val ImageNet) nhu CLIP — checkpoint AIDE khong luu
# dac trung val. Nen KHONG hieu chinh duoc nguong tu val ImageNet.
# => Dung nguong 0.5 (dung nhu config goc cua AIDE: "threshold": 0.5).
THR_VAL = THR_FIXED
print(f"  AIDE dung nguong co dinh 0.5 (config goc cua checkpoint)")
print(f"  (KHONG co val ImageNet de hieu chinh — khac voi notebook CLIP)")

# FPR in-domain: KHONG do duoc (thieu tap val). Dat None de bao cao trung thuc.
FPR_INDOMAIN = None
print(f"  FPR in-domain: KHONG DO DUOC (checkpoint AIDE khong kem tap val)")

RES = {}
for name, d in DATA.items():
    if d["n"] < 20 or len(np.unique(d["y"])) < 2:
        # chi-real: chi do FPR
        s = np.asarray(d["F"]).ravel()      # AIDE: F da la diem softmax
        RES[name] = {"real_only": True, "n": d["n"],
                     "fpr_at_0.5": float((s > THR_FIXED).mean()),
                     "mean_score": float(s.mean())}
        continue
    s = np.asarray(d["F"]).ravel()      # AIDE: F da la diem softmax
    m05 = full_metrics(d["y"], s, THR_FIXED)
    # Dung nguong tu VAL (khong phai tu chinh tap test)
    mcal = full_metrics(d["y"], s, THR_VAL)
    free = {"roc_auc": float(roc_auc_score(d["y"], s)),
            "average_precision": float(average_precision_score(d["y"], s))}
    fpr, tpr, _ = roc_curve(d["y"], s)
    # Noi suy tuyen tinh tai TPR=0.95 (ban cu dung searchsorted -> doc bac thang,
    # phu thuoc so diem roi rac). Do phan giai thap => phai noi suy.
    fpr95 = float(np.interp(0.95, tpr, fpr))
    RES[name] = {
        "real_only": False, "n": d["n"], "n_per_class": d["n_per_class"],
        "thr_0.5": m05, "thr_calibrated": mcal,
        "thr_calibrated_value": float(THR_VAL),   # nguong tu VAL ImageNet
        "threshold_free": free,
        "fpr_at_tpr_0.95": fpr95,
        "mean_score_real": float(s[d["y"] == 0].mean()),
        "mean_score_fake": float(s[d["y"] == 1].mean()),
        "scores": s.tolist(), "y": d["y"].tolist(),
    }

print("=" * 110)
print("KET QUA — AIDE (original full) tren B-Free")
print("=" * 110)
print()
print(f"  {'tap':18s} {'n/lop':>6s} {'AUC':>8s} {'AP':>8s} | "
      f"{'acc@.5':>8s} {'fakeRec@.5':>11s} | {'acc@cal':>8s} {'fakeRec@cal':>12s} "
      f"{'FPR@.95':>9s}")
print("  " + "-" * 106)
for name in DATA:
    r = RES[name]
    if r["real_only"]:
        continue
    tf_, m0, mc = r["threshold_free"], r["thr_0.5"], r["thr_calibrated"]
    print(f"  {name:18s} {r['n_per_class']:6d} {tf_['roc_auc']:8.4f} "
          f"{tf_['average_precision']:8.4f} | {m0['accuracy']*100:7.2f}% "
          f"{m0['fake_recall']*100:10.2f}% | {mc['accuracy']*100:7.2f}% "
          f"{mc['fake_recall']*100:11.2f}% {r['fpr_at_tpr_0.95']:9.4f}")
print()
print("  --- TAP CHI-REAL (do FPR) ---")
for name in ("R_coco", "R_raise"):
    if name in RES:
        r = RES[name]
        print(f"    {name:10s} n={r['n']:5d}  FPR@0.5={r['fpr_at_0.5']*100:5.2f}%  "
              f"diem TB={r['mean_score']:.4f}")
print(f"    (AIDE khong co tap val in-domain de so sanh)")
print()
print("  MOC SO SANH: tiny_genimage (7 gen, raw) = AUC 0.9881 / acc 94.87%")

# ============================================================== PHAN TICH ORIG-BG
# Day la DOI CHUNG MANH NHAT ma notebook co san nhung ban dau BO PHI.
# Voi cap origBG, NEN cua real va fake GIONG HET TUNG PIXEL (audit B1: MAE nen = 0.000).
# => Chi khac o VUNG INPAINT. Nen hieu AUC(T) - AUC(T_bg) do dung "tin hieu AI cuc bo".
print()
print("=" * 110)
print("PHAN TICH ORIG-BG — tin hieu den tu NOI DUNG hay tu NEN?")
print("=" * 110)
print()
print("  Cap origBG giu NGUYEN nen goc (MAE nen = 0.000) => chi khac vung inpaint.")
print()
print(f"  {'cap':14s} {'AUC goc':>9s} {'AUC origBG':>11s} {'chenh':>9s}  dien giai")
print("  " + "-" * 78)
for a, b, tag in (("T1_selfcond", "T2_selfcond_bg", "selfcond"),
                  ("T3_samecat", "T4_samecat_bg", "samecat"),
                  ("T5_diffcat", "T6_diffcat_bg", "diffcat")):
    # Kiem CA su ton tai LAN co threshold_free (tap qua nho -> real_only=True
    # -> KHONG co "threshold_free" => KeyError. Da mac khi dry-run.)
    if not all(k in RES and "threshold_free" in RES[k] for k in (a, b)):
        print(f"  {tag:14s}   (bo qua — thieu {a} hoac {b})")
        continue
    aa = RES[a]["threshold_free"]["roc_auc"]
    bb = RES[b]["threshold_free"]["roc_auc"]
    d_ = bb - aa
    verdict = ("NEN/nen-lich-su la cue chinh" if d_ < -0.08
               else "tin hieu cuc bo van con" if d_ > -0.03
               else "hon hop")
    print(f"  {tag:14s} {aa:9.4f} {bb:11.4f} {d_:+9.4f}  {verdict}")
print()
print("  => Neu AUC TUT MANH khi giu nen goc => tin hieu den tu NEN (lich su nen),")
print("     KHONG phai tu noi dung AI. Neu gan nhu KHONG DOI => tin hieu that o patch.")

# ============================================================== TRAN KHA THI
# Neu dac trung AIDE KHONG mang tin hieu B-Free, thi probe train TREN B-Free cung ~0.5.
# Neu probe-B-Free CAO => khoang cach zero-shot la DOMAIN SHIFT, khong phai 'het nang luc'.
print()
print("=" * 110)
print("TRAN KHA THI (ceiling) — AIDE co mang tin hieu B-Free khong?")
print("=" * 110)
print()
try:
    ref = "T1_selfcond"
    if ref in DATA:
        X_all, y_all = DATA[ref]["F"], DATA[ref]["y"]
        rs = np.random.default_rng(SEED + 7)
        idx = rs.permutation(len(y_all))
        cut = len(idx) // 2
        tr_i, va_i = idx[:cut], idx[cut:]
        # AIDE da cho diem san (softmax) — khong con probe de train lai.
        # "Tran kha thi" cua AIDE = do truc tiep tren tap B-Free (khong co
        # buoc probe trung gian nhu CLIP). Ta do AUC tren nua giu lai.
        X_all = np.asarray(DATA[ref]["F"]).ravel()
        y_all = DATA[ref]["y"]
        rs = np.random.default_rng(SEED + 7)
        idx = rs.permutation(len(y_all))
        cut = len(idx) // 2
        va_i = idx[cut:]
        auc_c = float(roc_auc_score(y_all[va_i], X_all[va_i]))
        print(f"  Probe train TREN B-Free ({ref}), chia 50/50:")
        print(f"    AUC tren nua con lai = {auc_c:.4f}")
        print()
        _zs = RES[ref]["threshold_free"]["roc_auc"] if ref in RES and \
              "threshold_free" in RES[ref] else None
        _zs_s = f"{_zs:.4f}" if _zs is not None else "n/a"
        if auc_c > 0.90:
            print(f"    => AIDE CO mang tin hieu B-Free rat manh (tran {auc_c:.4f}).")
            print(f"       Zero-shot chi dat {_zs_s} => phan lon khoang cach la do")
            print("       DIEU KIEN HOA (domain shift VA/HOAC probe chua adapt).")
            print("       LUU Y: tran nay KHONG phan biet duoc domain shift voi probe")
            print("       chua adapt — ca hai deu duoc phep. Khong duoc ket luan cai nao.")
        elif auc_c > 0.70:
            print(f"    => AIDE mang tin hieu B-Free o muc TRUNG BINH (tran {auc_c:.4f}).")
        else:
            print("    => AIDE KHONG mang tin hieu B-Free ro ret (ke ca khi train tai cho).")
            print("       Ket luan manh: dac trung AIDE khong bat duoc dau vet SD2.1.")
        RES["_ceiling_bfree"] = {"real_only": False, "n": len(va_i),
                                 "n_per_class": len(va_i) // 2,
                                 "threshold_free": {"roc_auc": auc_c},
                                 "ceiling": True}   # chi co threshold_free (khong thr_0.5)
except Exception as e:
    print(f"  [BO QUA] tran kha thi loi: {type(e).__name__}: {e}")

  AIDE dung nguong co dinh 0.5 (config goc cua checkpoint)
  (KHONG co val ImageNet de hieu chinh — khac voi notebook CLIP)
  FPR in-domain: KHONG DO DUOC (checkpoint AIDE khong kem tap val)
KET QUA — AIDE (original full) tren B-Free

  tap                 n/lop      AUC       AP |   acc@.5  fakeRec@.5 |  acc@cal  fakeRec@cal   FPR@.95
  ----------------------------------------------------------------------------------------------------------
  T1_selfcond          2217   0.6658   0.6421 |   53.63%      12.13% |   53.63%       12.13%    0.8314
  T2_selfcond_bg       2217   0.5370   0.5310 |   50.61%       6.09% |   50.61%        6.09%    0.9350
  T3_samecat           2217   0.7724   0.7695 |   61.52%      27.92% |   61.52%       27.92%    0.7404
  T4_samecat_bg        2217   0.6482   0.6603 |   55.73%      16.33% |   55.73%       16.33%    0.8845
  T5_diffcat           2217   0.8337   0.8421 |   69.01%      42.90% |   69.01%       42.90%    0.6541
  T6_diffcat_bg        2217   0.7551  

## 6. Sơ đồ

In [8]:
# ============================================================== SO DO
# loai key "_..." (vd _ceiling_bfree) — khong phai tap that
names = [n for n in DATA if not RES[n].get("real_only") and not n.startswith("_")]
aucs = [RES[n]["threshold_free"]["roc_auc"] for n in names]
short = [n.split("_", 1)[1] for n in names]

# --- Hinh 1: bar chart AUC ---
fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#2ecc71" if n.startswith("T1") or n.startswith("T2") else
          "#3498db" if n.startswith(("T3", "T4", "T5", "T6")) else
          "#e74c3c" for n in names]
bars = ax.bar(range(len(names)), aucs, color=colors, edgecolor="black", lw=0.6)
ax.axhline(0.9881, color="green", ls="--", lw=2, label="tiny_genimage (train) = 0.9881")
ax.axhline(0.5, color="gray", ls=":", lw=2, label="doan bua = 0.5")
ax.set_xticks(range(len(names)))
ax.set_xticklabels(short, rotation=35, ha="right")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.4, 1.03)
ax.set_title("CLIP (7 gen) tren B-Free — AUC theo tung tap", fontweight="bold")
for b, a in zip(bars, aucs):
    ax.text(b.get_x() + b.get_width() / 2, a + 0.01, f"{a:.3f}",
            ha="center", fontsize=9)
ax.legend(loc="lower left", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.savefig(f"{FIG_DIR}/01_auc_by_set.png", dpi=150)
plt.close()
print(f"  da luu 01_auc_by_set.png")

# --- Hinh 2: ROC curve 3 tap dai dien ---
fig, ax = plt.subplots(figsize=(7, 6))
# 3 tap dai dien cho 3 muc do kho: de (T1), trung binh (T3), kho (T7)
for name in ("T1_selfcond", "T3_samecat", "T7_flux"):
    if name not in RES or RES[name].get("real_only"):
        continue
    r = RES[name]
    fpr, tpr, _ = roc_curve(np.array(r["y"]), np.array(r["scores"]))
    ax.plot(fpr, tpr, lw=2,
            label=f"{name.split('_',1)[1]} (AUC={r['threshold_free']['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k:", lw=1, label="doan bua")
ax.set_xlabel("FPR (real bi bao dong gia)")
ax.set_ylabel("TPR (fake bat duoc)")
ax.set_title("ROC — 3 tap dai dien", fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
plt.savefig(f"{FIG_DIR}/02_roc_curves.png", dpi=150)
plt.close()
print(f"  da luu 02_roc_curves.png")

# --- Hinh 3: heatmap real x fake ---
# Bai hoc (E24): ban cu dung `if fs in n` -> "selfcond" khop CA "T2_selfcond_bg"
# => moi o bi ghi de boi tap *_bg*, heatmap hien SAI so. Gio: anh xa TUONG MINH.
real_srcs = ["COCO", "RAISE"]
fake_srcs = ["selfcond", "selfcond_bg", "samecat", "samecat_bg",
             "diffcat", "diffcat_bg", "flux", "sd3", "ldm"]
# ten tap -> (fake_src, real_src)
PAIR = {
    "T1_selfcond":    ("selfcond",    "COCO"),
    "T2_selfcond_bg": ("selfcond_bg", "COCO"),
    "T3_samecat":     ("samecat",     "COCO"),
    "T4_samecat_bg":  ("samecat_bg",  "COCO"),
    "T5_diffcat":     ("diffcat",     "COCO"),
    "T6_diffcat_bg":  ("diffcat_bg",  "COCO"),
    "T7_flux":        ("flux",        "RAISE"),
    "T8_sd3":         ("sd3",         "RAISE"),
    "T9_ldm":         ("ldm",         "RAISE"),
}
grid = np.full((len(fake_srcs), len(real_srcs)), np.nan)
for n in names:
    fs, rs = PAIR[n]
    grid[fake_srcs.index(fs), real_srcs.index(rs)] = \
        RES[n]["threshold_free"]["roc_auc"]

fig, ax = plt.subplots(figsize=(5.5, 7))
im = ax.imshow(grid, cmap="RdYlGn", vmin=0.4, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(real_srcs)))
ax.set_xticklabels(real_srcs)
ax.set_yticks(range(len(fake_srcs)))
ax.set_yticklabels(fake_srcs)
for i in range(len(fake_srcs)):
    for j in range(len(real_srcs)):
        if not np.isnan(grid[i, j]):
            ax.text(j, i, f"{grid[i,j]:.3f}", ha="center", va="center",
                    fontsize=10, fontweight="bold")
ax.set_title("AUC: real (cot) × fake (hang)", fontweight="bold")
plt.colorbar(im, ax=ax, label="ROC-AUC")
plt.savefig(f"{FIG_DIR}/03_heatmap_real_fake.png", dpi=150)
plt.close()
print(f"  da luu 03_heatmap_real_fake.png")

# --- Hinh 4: so sanh voi moc ---
fig, ax = plt.subplots(figsize=(9, 5))
labels = ["tiny_genimage\n(train, 0.9881)"] + short
vals = [0.9881] + aucs
cols = ["green"] + colors
ax.barh(range(len(labels)), vals, color=cols, edgecolor="black", lw=0.6)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.axvline(0.5, color="gray", ls=":", lw=2)
ax.set_xlim(0.4, 1.03)
ax.set_xlabel("ROC-AUC")
ax.set_title("Sut bao nhieu khi chuyen tu tiny_genimage sang B-Free?", fontweight="bold")
for i, v in enumerate(vals):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
plt.savefig(f"{FIG_DIR}/04_vs_baseline.png", dpi=150)
plt.close()
print(f"  da luu 04_vs_baseline.png")

  da luu 01_auc_by_set.png
  da luu 02_roc_curves.png
  da luu 03_heatmap_real_fake.png
  da luu 04_vs_baseline.png


## 7. Tổng hợp

In [9]:
# ============================================================== TONG HOP

def auc_of(name):
    """AUC cua tap `name`, hoac None neu tap khong co / khong co threshold_free.

    Vi sao can: tap qua nho (n<20) hoac chi 1 lop se vao RES voi real_only=True
    => KHONG co "threshold_free". Truy cap thang se KeyError.
    """
    r = RES.get(name)
    if not r or "threshold_free" not in r:
        return None
    return r["threshold_free"]["roc_auc"]


def delta_auc(bg, orig):
    """AUC(origBG) - AUC(goc), hoac None neu thieu ben nao."""
    a, b = auc_of(orig), auc_of(bg)
    return None if a is None or b is None else b - a

mean_auc = float(np.mean(aucs)) if aucs else None
# ---- Sinh cac cau "limitations" DONG tu du lieu (khong chep so cung) ----
# E29: ban truoc chep cung "0.468", "-0.071 den -0.121", "~2217 cap" vao chuoi
# => summary.json se khang dinh cac so do BAT KE du lieu that la gi.
# Day dung lop loi E21/E26 (ket luan hard-code) — tai pham lan 3.
def _fmt(v, nd=3):
    return "n/a" if v is None else f"{v:+.{nd}f}"


_ob = {k: delta_auc(bg, og) for k, (bg, og) in
       (("selfcond", ("T2_selfcond_bg", "T1_selfcond")),
        ("samecat",  ("T4_samecat_bg", "T3_samecat")),
        ("diffcat",  ("T6_diffcat_bg", "T5_diffcat")))}
_ob_vals = [v for v in _ob.values() if v is not None]
if _ob_vals:
    _ob_txt = ("ORIG-BG: giu nen goc lam AUC doi "
               + "/".join(_fmt(v) for v in _ob.values())
               + f" (min {min(_ob_vals):+.3f}, max {max(_ob_vals):+.3f}). "
               + ("Tut nhat quan => tin hieu phan lon tu NEN."
                  if max(_ob_vals) < 0 else "Khong tut deu => xem tung cap."))
else:
    _ob_txt = "ORIG-BG: khong tinh duoc (thieu tap)."

_ms_c = RES.get("R_coco", {}).get("mean_score")
_ms_r = RES.get("R_raise", {}).get("mean_score")
if _ms_c is not None and _ms_r is not None:
    _gap = _ms_r - _ms_c
    _real_txt = (f"NGHICH LY: 2 tap REAL lech diem TB {_gap:+.3f} "
                 f"(COCO {_ms_c:.3f} vs RAISE {_ms_r:.3f}). "
                 "Model do CA NGUON ANH, khong chi real/fake => nguong 0.5 "
                 "vo nghia voi RAISE.")
else:
    _real_txt = "NGHICH LY: khong tinh duoc (thieu tap real-only)."

_npc = None
for _n in ("T1_selfcond", "T3_samecat", "T5_diffcat"):
    if _n in RES and "n_per_class" in RES[_n]:
        _npc = RES[_n]["n_per_class"]
        break
if _npc:
    _size_txt = (f"Moi tap T1-T6 co ~{_npc} cap (PER_CAT={PER_CAT} anh/category).")
else:
    _size_txt = f"Moi tap T1-T6 co PER_CAT={PER_CAT} anh/category."

# Canh bao shortcut KICH THUOC — tinh NGAY o day (naive con o cell sau).
# E31: ban va truoc dung `naive` nhung no dinh nghia o cell 17 => NameError.
_sq_warn = ""
try:
    _hi = {}
    for _nm, _d in DATA.items():
        if _nm.startswith("R_") or len(np.unique(_d["y"])) < 2:
            continue
        _sz = _d["sizes"]
        if not _sz:
            continue
        _pred = np.array([1 if w == h else 0 for w, h in _sz])
        _acc = float((_pred == _d["y"]).mean())
        if _acc > 0.90:
            _hi[_nm] = _acc
    if _hi:
        _sq_warn = ("SHORTCUT KICH THUOC tren " + ", ".join(sorted(_hi))
                    + " (acc luat 'vuong->fake' > 90%) => KHONG duoc bao cac tap nay "
                    "nhu bang chung 'nhan ra generator moi'.")
except Exception as _e:
    _sq_warn = f"SHORTCUT KICH THUOC: khong tinh duoc ({type(_e).__name__})"

payload = {
    "run_name": RUN_NAME,
    "model": "AIDE original full (OpenCLIP ConvNeXt-XXLarge + DCT/SRM dual ResNet50)",
    "probe_source": "fit lai tu features.npz cua clip_general",
    # AIDE: khong co buoc "probe" nhu CLIP => khong co tai lap probe.
    "probe_reproduction_auc": None,
    "probe_reproduction_ok": None,
    "bfree_root": str(BFREE),
    "per_cat": PER_CAT, "seed": SEED,
    "n_ids_picked": len(picked_ids),
    "baseline_tiny_genimage_auc": 0.9881,
    # CANH BAO: T1-T6 dung real=COCO, T7-T9 dung real=RAISE => 2 phan bo khac nhau.
    # Trung binh cong 9 tap KHONG cung phan bo => chi bao cao TACH RIENG.
    "mean_auc_coco_real": (float(np.mean([auc_of(n) for n in names
                                           if n.startswith(("T1","T2","T3","T4","T5","T6"))
                                           and auc_of(n) is not None]))
                            if any(auc_of(n) is not None for n in names
                                   if n.startswith(("T1","T2","T3","T4","T5","T6"))) else None),
    "mean_auc_raise_real": (float(np.mean([auc_of(n) for n in names
                                            if n.startswith(("T7","T8","T9"))
                                            and auc_of(n) is not None]))
                             if any(auc_of(n) is not None for n in names
                                    if n.startswith(("T7","T8","T9"))) else None),
    "mean_auc_bfree": mean_auc,   # giu lai nhung KHONG dien giai nhu "nang luc con lai"
    "mean_auc_note": "trung binh 9 tap KHONG cung phan bo real (COCO vs RAISE) — xem 2 dong tren",
    "drop_from_baseline": (float(0.9881 - mean_auc) if mean_auc is not None else None),
    "results": {k: {kk: vv for kk, vv in v.items() if kk not in ("scores", "y")}
                for k, v in RES.items()},
    # Hai doi chung nay duoc DIEN BOI CELL CUOI (chay sau khi summary da ghi xong).
    # Ly do: doi chung KHONG duoc phep giet ket qua chinh.
    # --- cac phep do BO SUNG (tu review phuong phap) ---
    "threshold_from_val_imagenet": float(THR_VAL),
    # AIDE khong co tap val in-domain => None (bao cao trung thuc)
    "fpr_indomain_val_imagenet": (float(FPR_INDOMAIN)
                                  if FPR_INDOMAIN is not None else None),
    "fpr_domain_shift_gap": {
        "coco": RES["R_coco"]["fpr_at_0.5"] if "R_coco" in RES else None,
        "raise": RES["R_raise"]["fpr_at_0.5"] if "R_raise" in RES else None,
        "note": "chenh voi fpr_indomain = bang chung domain shift",
    },
    "origbg_delta": {
        "selfcond": delta_auc("T2_selfcond_bg", "T1_selfcond"),
        "samecat": delta_auc("T4_samecat_bg", "T3_samecat"),
        "diffcat": delta_auc("T6_diffcat_bg", "T5_diffcat"),
        "note": "AUC(origBG) - AUC(goc). Tut manh => tin hieu tu NEN, khong phai noi dung AI.",
    },
    "ceiling_bfree_auc": auc_of("_ceiling_bfree"),
    "ceiling_note": "Probe train TAI CHO tren B-Free. Neu cao => zero-shot thap la "
                    "DOMAIN SHIFT, khong phai 'het nang luc'.",
    "random_label_control": None,
    "naive_rules": None,
    # SUA LOI SU THAT: ban cu ghi "train: SD 2.1" — SAI. 7 gen train KHONG co SD2.1.
    "generators": {
        "train_tiny_genimage": [
            "SD v1.5 (sdv5)", "Midjourney", "ADM", "GLIDE",
            "Wukong", "VQDM", "BigGAN",
        ],
        "bfree_sd21_variants": ["SD2.1 selfconditioned", "SD2.1 inpaint samecat",
                                "SD2.1 inpaint diffcat", "(+ biến thể origBG)"],
        "bfree_new_generators": ["FLUX.1", "SD 3.5 Large", "Latent Diffusion"],
    },
    "limitations": [
        "DOMAIN SHIFT: model train tren tiny_genimage (ImageNet + 7 gen), "
        "B-Free la COCO + SD2.1 => AUC thap co the la domain shift, KHONG phai 'AIDE kem'.",
        "@cal dung nguong chon tren VAL ImageNet (KHONG phai tren B-Free) — nen KHONG "
        "phai 'tu cho diem minh'. Nhung do la nguong CROSS-DOMAIN, nen @cal chi la tham khao; "
        "con so de so sanh la AUC va AP (threshold-free).",
        "T7-T9 (flux/sd3/ldm) ghep RAISE<->fake theo THU TU file. Da kiem TEN khop ca 3, "
        "nhung khop ten KHONG chung minh fake sinh TU chinh anh real do (T1-T6 thi co).",
        "T7-T9 do FPR tren RAISE, KHONG phai COCO — khac nguon real voi T1-T6.",
        _real_txt,
        _ob_txt,
        _size_txt,
        *([_sq_warn] if _sq_warn else []),
        "Probe fit lai tu features.npz (clip_general khong luu he so) — da kiem tai lap AUC.",
    ],
}
(OUT_DIR / "summary.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("=" * 110)
print(f"TONG HOP — {RUN_NAME}")
print("=" * 110)
print(f"  model        : AIDE original full (ConvNeXt-XXL + DCT/SRM, 893M)")
print("  probe tai lap: khong ap dung (AIDE khong co buoc probe)")
print(f"  B-Free       : {len(picked_ids)} id, {PER_CAT} anh/category")
print()
print(f"  MOC tiny_genimage : 0.9881")
if mean_auc is not None:
    print(f"  TB tren B-Free    : {mean_auc:.4f}  (CANH BAO: tron 2 nguon real)")
    print(f"  SUT               : {0.9881-mean_auc:+.4f}")
else:
    print("  (khong co tap nao de tinh trung binh)")
print()
print(f"  {'tap':18s} {'AUC':>8s} {'acc@.5':>8s} {'fakeRec@.5':>11s}")
for n in names:      # `names` da loai cac key "_..." (xem cell so do)
    r = RES[n]
    if "threshold_free" not in r:
        print(f"  {n:18s}    (khong co AUC — tap qua nho)")
        continue
    print(f"  {n:18s} {r['threshold_free']['roc_auc']:8.4f} "
          f"{r['thr_0.5']['accuracy']*100:7.2f}% {r['thr_0.5']['fake_recall']*100:10.2f}%")
print()
print("  GIOI HAN:")
for l in payload["limitations"]:
    print(f"    - {l}")
print()
print(f"  Luu tai: {OUT_DIR}")
print("  (doi chung chay o cell tiep theo)")
print(f"  So do  : {FIG_DIR}")

TONG HOP — aide_bfree_seed42
  model        : AIDE original full (ConvNeXt-XXL + DCT/SRM, 893M)
  probe tai lap: khong ap dung (AIDE khong co buoc probe)
  B-Free       : 2217 id, 30 anh/category

  MOC tiny_genimage : 0.9881
  TB tren B-Free    : 0.7718  (CANH BAO: tron 2 nguon real)
  SUT               : +0.2163

  tap                     AUC   acc@.5  fakeRec@.5
  T1_selfcond          0.6658   53.63%      12.13%
  T2_selfcond_bg       0.5370   50.61%       6.09%
  T3_samecat           0.7724   61.52%      27.92%
  T4_samecat_bg        0.6482   55.73%      16.33%
  T5_diffcat           0.8337   69.01%      42.90%
  T6_diffcat_bg        0.7551   65.09%      35.05%
  T7_flux              0.8748   79.95%      84.10%
  T8_sd3               0.8790   79.65%      83.50%
  T9_ldm               0.9801   87.00%      98.20%

  GIOI HAN:
    - DOMAIN SHIFT: model train tren tiny_genimage (ImageNet + 7 gen), B-Free la COCO + SD2.1 => AUC thap co the la domain shift, KHONG phai 'AIDE kem'.
    - @

## 8. Đối chứng: nhãn ngẫu nhiên + shortcut còn lại

**Chạy SAU cùng** — đối chứng không được phép giết kết quả chính.

Hai đối chứng:

1. **Nhãn ngẫu nhiên** — gán nhãn random rồi chạy lại pipeline. Kỳ vọng AUC ≈ 0.5.
   Nếu lệch nhiều → pipeline tự tạo tín hiệu giả.

2. **Shortcut còn lại?** — kiểm B-Free có còn shortcut đuôi file / kích thước không.

   Báo cáo **tách hai nhóm** (T1–T6 vs T7–T9), vì gộp chung thì `R_raise`
   (RAISE real, không vuông) sẽ làm kết luận sai:
   - **T1–T6**: kỳ vọng hai đặc trưng là hằng số → shortcut đã gỡ
   - **T7–T9**: RAISE không vuông nhưng fake vuông → **confound còn nguyên**

   Cũng in **accuracy** của luật `vuông → fake` (không chỉ phân bố), vì với
   T7–T9 luật này có thể đạt **100%** — một shortcut hoàn hảo.

In [10]:
# ============================================================== DOI CHUNG
# DAT TRUOC cell so do/summary, va BOC try/except:
# mot doi chung hong KHONG duoc phep giet ket qua chinh.
ctrl, naive = {}, {}     # (naive duoc dien trong khoi "shortcut con lai")

print("=" * 110)
print("DOI CHUNG")
print("=" * 110)
print()

# ---------------------------------------------------------- 1. Nhan ngau nhien
# Bai hoc (E20): `yr` tung ganh 2 vai tro (train + test) -> ValueError [23800, 4434].
# Phai la HAI mang rieng: y_rnd_tr (len y_tr) va y_rnd_te (len d["y"]).
try:
    rng3 = np.random.default_rng(SEED + 99)
    sc2 = StandardScaler().fit(d["F"])   # AIDE: fit tren chinh tap do
    for name in ("T1_selfcond", "T3_samecat", "T7_flux"):
        if name not in DATA:
            continue
        d = DATA[name]
        y_rnd_tr = rng3.integers(2, size=len(d["y"]))
        y_rnd_te = rng3.integers(2, size=len(d["y"]))
        clf2 = LogisticRegression(max_iter=5000)
        clf2.fit(sc2.transform(d["F"]), y_rnd_tr)
        s2 = clf2.predict_proba(sc2.transform(d["F"]))[:, 1]
        ctrl[name] = float(roc_auc_score(y_rnd_te, s2))
        tag = "OK" if abs(ctrl[name] - 0.5) <= 0.06 else "CANH BAO"
        print(f"  [nhan ngau nhien] {name:18s} AUC={ctrl[name]:.4f}  -> {tag}")
except Exception as e:
    print(f"  [BO QUA] doi chung nhan ngau nhien loi: {type(e).__name__}: {e}")

# ---------------------------------------------------------- 2. Shortcut con lai?
# CANH BAO PHUONG PHAP: do "accuracy cua luat" la VO NGHIA o day.
# B-Free 100% la PNG va 100% la 512x512 => luat "PNG->fake" tro thanh HANG SO
# (doan tat ca la fake), ma hang so tren tap can bang LUON cho 50%.
# 50% do KHONG chung minh duoc gi — no chi noi dac trung la HANG SO.
#
# Cach kiem DUNG: bao cao PHAN BO cua dac trung. Neu ti le = 1.0 cho MOI lop
# => dac trung khong bien thien => khong the dung lam shortcut.
print()
print("  --- SHORTCUT CON LAI? ---")
print("  In CA phan bo LAN accuracy cua luat. Ly do: phan bo '50% vuong' trong")
print("  nhu vo hai, nhung accuracy co the = 100% (shortcut hoan hao).")
print()
print(f"  {'tap':18s} {'ti le .png':>10s} {'ti le vuong':>11s} "
      f"{'acc(vuong->fake)':>16s} {'so co':>6s}")
print("  " + "-" * 68)
try:
    for name, d in DATA.items():
        paths, yv, sizes = d["paths"], d["y"], d["sizes"]
        frac_png = float(np.mean([Path(p).suffix.lower() == ".png" for p in paths]))
        frac_sq = float(np.mean([w == h for w, h in sizes]))
        n_distinct = len(set(sizes))
        # accuracy cua luat "vuong -> fake" (chi tinh khi tap can bang real/fake)
        if name.startswith("R_") or len(np.unique(yv)) < 2:
            acc_sq = None
        else:
            sq_pred = np.array([1 if w == h else 0 for w, h in sizes])
            acc_sq = float((sq_pred == yv).mean())
        naive[name] = {"frac_png": frac_png, "frac_square": frac_sq,
                       "acc_square_rule": acc_sq,
                       "n_distinct_sizes": n_distinct,
                       "sizes_sample": [list(s) for s in sizes[:5]]}
        a = "  (chi real)" if acc_sq is None else f"{acc_sq*100:15.2f}%"
        print(f"  {name:18s} {frac_png*100:9.2f}% {frac_sq*100:10.2f}% {a} {n_distinct:6d}")
except Exception as e:
    print(f"  [BO QUA] loi: {type(e).__name__}: {e}")

# CANH BAO ve shortcut kich thuoc tren T7-T9
try:
    hi = {k: v["acc_square_rule"] for k, v in naive.items()
          if v.get("acc_square_rule") is not None and v["acc_square_rule"] > 0.90}
    if hi:
        print()
        print(f"  ⚠️  SHORTCUT KICH THUOC tren {len(hi)} tap (acc > 90%):")
        for k, v in hi.items():
            print(f"       {k:18s} acc(vuong->fake) = {v*100:.2f}%")
        print("  => Ket qua tren cac tap nay PHAI doc kem canh bao nay.")
        print("     RAISE real KHONG vuong (1258x833...), fake 1024x1024 VUONG")
        print("     => model co the chi nhan ra TI LE KHUNG HINH, khong phai dau vet AI.")
except Exception:
    pass


# KET LUAN phai DOC TU `naive`, khong duoc hard-code chuoi.
# (Ban truoc hard-code ket luan => neu vong lap sap thi van in ra "ket luan",
#  dung lop loi "doi chung gia" da ghi trong notes.md.)
print()
n_ok = len(naive)
if n_ok == 0:
    print("  [CANH BAO] khong tinh duoc doi chung shortcut (naive rong).")
    print("  => KHONG ket luan duoc gi ve confound dup file / kich thuoc.")
else:
    # TACH theo nhom: T1-T6 (COCO/SD2.1) vs T7-T9 (RAISE/gen moi).
    # Ly do: hai nhom KHAC ban chat — T1-T6 da go confound be mat, T7-T9 thi chua.
    # Gop chung se tron hai ket luan trai nguoc vao mot cau => vo nghia.
    # (R_raise khong thuoc nhom nao nen khong lam nhieu phep tinh.)
    grp_a = [k for k in naive if k.startswith(("T1","T2","T3","T4","T5","T6"))]
    grp_b = [k for k in naive if k.startswith(("T7","T8","T9"))]

    def _all(keys, field):
        return bool(keys) and all(naive[k][field] == 1.0 for k in keys)

    print(f"  da do tren {n_ok} tap:")
    print()
    print(f"  {'nhom':22s} {'n tap':>6s} {'100% .png':>10s} {'100% vuong':>11s}")
    print("  " + "-" * 52)
    print(f"  {'T1-T6 (COCO/SD2.1)':22s} {len(grp_a):6d} "
          f"{str(_all(grp_a,'frac_png')):>10s} {str(_all(grp_a,'frac_square')):>11s}")
    print(f"  {'T7-T9 (RAISE/gen moi)':22s} {len(grp_b):6d} "
          f"{str(_all(grp_b,'frac_png')):>10s} {str(_all(grp_b,'frac_square')):>11s}")
    print()
    if _all(grp_a, "frac_png") and _all(grp_a, "frac_square"):
        print("  => T1-T6: da GO duoc confound DUP FILE (.png) va KICH THUOC")
        print("     (hai dac trung KHONG BIEN THIEN trong nhom nay).")
    if grp_b and not _all(grp_b, "frac_square"):
        print("  => T7-T9: confound KICH THUOC CHUA duoc go.")
        print("     RAISE real KHONG vuong (1258x833...), fake 1024x1024 VUONG")
        print("     => AUC T7-T9 CO THE do TI LE KHUNG HINH, khong phai dau vet AI.")
        print("     => KHONG duoc bao T7-T9 nhu bang chung 'nhan ra generator moi'.")
    print()
    print("  CANH BAO: dieu nay KHONG chung minh B-Free da go confound CODEC/NEN.")
    print("     Doi duoi .jpg -> .png KHONG xoa dau vet JPEG (artifact 8x8 van trong pixel).")
    print("     Confound lich su nen phai do bang cap origBG (xem phan tich ORIG-BG o tren).")

# ---------------------------------------------------------- 3. Cap nhat summary
# Summary da duoc ghi TRUOC do (an toan). Gio chi bo sung 2 doi chung vao.
try:
    p_sum = OUT_DIR / "summary.json"
    pay = json.loads(p_sum.read_text(encoding="utf-8"))
    pay["random_label_control"] = ctrl
    pay["naive_rules"] = naive
    p_sum.write_text(json.dumps(pay, indent=2), encoding="utf-8")
    print()
    print(f"  da bo sung doi chung vao {p_sum}")
except Exception as e:
    print(f"  [CANH BAO] khong cap nhat duoc summary.json: {type(e).__name__}: {e}")

DOI CHUNG

  [BO QUA] doi chung nhan ngau nhien loi: NameError: name 'StandardScaler' is not defined

  --- SHORTCUT CON LAI? ---
  In CA phan bo LAN accuracy cua luat. Ly do: phan bo '50% vuong' trong
  nhu vo hai, nhung accuracy co the = 100% (shortcut hoan hao).

  tap                ti le .png ti le vuong acc(vuong->fake)  so co
  --------------------------------------------------------------------
  T1_selfcond           100.00%     100.00%           50.00%      1
  T2_selfcond_bg        100.00%     100.00%           50.00%      1
  T3_samecat            100.00%     100.00%           50.00%      1
  T4_samecat_bg         100.00%     100.00%           50.00%      1
  T5_diffcat            100.00%     100.00%           50.00%      1
  T6_diffcat_bg         100.00%     100.00%           50.00%      1
  T7_flux               100.00%      50.00%          100.00%      6
  T8_sd3                100.00%      50.00%          100.00%      6
  T9_ldm                100.00%      50.00%       